# Colab OCR Batch Runner

Use this notebook to run heavy OCR batches in Google Colab instead of the local machine. It reuses raw OCR JSON when present, so runtime resets do not force a full restart.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/content/drive/MyDrive/election2026/ProjectDSDE_election2026')
os.chdir(REPO_DIR)
print(Path.cwd())

## Prepare PDFs

If the PDFs are already inside `data/raw/pdfs/extracted/`, leave `PDF_ZIP` empty. If you keep the prepared zip in Drive, set the path and run the cell.

In [ ]:
import zipfile

PDF_ZIP = Path('')  # Example: Path('/content/drive/MyDrive/election2026/raw_pdfs/chaiyaphum_2.zip')
PDF_EXTRACT_DIR = Path('data/raw/pdfs/extracted')

if str(PDF_ZIP) and PDF_ZIP.exists():
    PDF_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PDF_ZIP) as archive:
        archive.extractall(PDF_EXTRACT_DIR)
    print(f'Extracted {PDF_ZIP} -> {PDF_EXTRACT_DIR}')
else:
    print('Skipped PDF unzip. Confirm manifest paths already exist before OCR.')

## Install dependencies

Run this once per fresh Colab runtime. Restart the runtime if PaddleOCR asks for it, then continue from the next cell.

In [ ]:
!pip -q install -r requirements.txt

## Check OCR progress

Use `manifest_index` to pick a small batch. Rows 1-4 are advance vote forms; rows 5+ are election-day PDFs.

In [ ]:
!python -m src.pipeline.ocr_progress --config configs/chaiyaphum_2.yaml

import pandas as pd
progress = pd.read_csv('data/processed/ocr_progress.csv')
progress[['manifest_index', 'form_type', 'file_name', 'pdf_pages', 'raw_json_pages', 'parsed_rows', 'status']].head(40)

## Run one OCR batch

Change `START_INDEX` and `END_INDEX` for each Colab run. Keep batches small if the runtime has limited memory.

In [ ]:
START_INDEX = 5
END_INDEX = 8

!python -m src.ocr.extract --config configs/chaiyaphum_2.yaml --start-index {START_INDEX} --end-index {END_INDEX}

## Rebuild outputs without rerunning OCR

In [ ]:
!python -m src.pipeline.run_all --config configs/chaiyaphum_2.yaml --skip-ocr
!python -m src.pipeline.ocr_progress --config configs/chaiyaphum_2.yaml

pd.read_csv('data/processed/validation_report.csv')

## Export artifacts back to Drive

Copy this zip back to the local repo root and unzip it. The local machine can then run `run_all --skip-ocr` without loading OCR models.

In [ ]:
from datetime import datetime
import zipfile

artifact_dir = Path('/content/drive/MyDrive/election2026/artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_path = artifact_dir / f'chaiyaphum2_ocr_batch_{START_INDEX}_{END_INDEX}_{stamp}.zip'

paths_to_zip = [
    Path('data/raw/ocr'),
    Path('data/processed/parsed'),
    Path('data/processed/ocr_progress.csv'),
    Path('data/processed/validation_report.csv'),
    Path('outputs/reports'),
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for item in paths_to_zip:
        if item.is_file():
            archive.write(item, item.as_posix())
        elif item.is_dir():
            for path in item.rglob('*'):
                if path.is_file():
                    archive.write(path, path.as_posix())

print(zip_path)